# W8 · Day 2 — Metadata, PII Scrubbing, Pipeline Wiring

**~90 minutes · in-class demo · Jupyter notebook · Track A**

Day 1 taught parsing and chunking. Day 2 covers what makes the pipeline
production-ready: **metadata** (for filtered retrieval) and **PII scrubbing**
(so sensitive data never reaches the vector store). We wrap by wiring
everything into a mini-pipeline on ONE sample document.

**This is the first appearance of the Security + PII recurring thread** —
returns in W14 (tool permissioning), W18 (prompt injection), W25 (observability
data), W28 (data residency deep-dive).

**Cost per full run:** ~$0.001 (a few embedding calls for the Qdrant demo).

**Notebook flow:**
- Cell 1: Setup + hard assertion that Presidio is installed (yesterday's homework)
- Cell 2: The 9-field metadata schema
- Cell 3: Qdrant filter DSL — build a demo collection, run filtered queries
- Cell 4: Why PII in vector stores is a discoverability problem
- Cells 5-8: PII detection — regex baseline, Presidio NER, anonymization, three-legs pattern
- Cell 9: Data residency — name your stack
- Cell 10: Wire it all together on ONE sample document
- Cell 11: Wrap + hand-off to Track B

---

## Cell 1 — Setup + Presidio check

Yesterday's homework: `pip install presidio-analyzer presidio-anonymizer` +
`python -m spacy download en_core_web_sm`.

The assertion below fails **hard** if Presidio isn't installed. That's
intentional — this notebook depends on it, and quietly skipping content is
worse than a loud stop.

In [ ]:
import os
import sys
from pathlib import Path

# ── Presidio check — HARD ASSERTION ──
try:
    from presidio_analyzer import AnalyzerEngine
    from presidio_anonymizer import AnonymizerEngine
except ImportError as e:
    raise ImportError(
        "\n\n"
        "═══════════════════════════════════════════════════════════════\n"
        "PRESIDIO NOT INSTALLED\n"
        "═══════════════════════════════════════════════════════════════\n"
        "This notebook requires Presidio (yesterday's homework).\n\n"
        "Run:\n"
        "   pip install presidio-analyzer presidio-anonymizer\n"
        "   python -m spacy download en_core_web_sm\n\n"
        "Total download: ~150 MB. Then restart the kernel and re-run.\n"
        "═══════════════════════════════════════════════════════════════\n"
    ) from e

# ── Other imports ──
assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY before running this notebook"
assert os.environ.get("QDRANT_URL"),     "Set QDRANT_URL — from W7 Qdrant Cloud setup"
assert os.environ.get("QDRANT_API_KEY"), "Set QDRANT_API_KEY — from W7 Qdrant Cloud setup"

# Import Day 1's helper module
sys.path.insert(0, str(Path.cwd()))
from wk08_pipeline import (
    SAMPLE_PDF, SAMPLE_HTML, SAMPLE_DOCX,
    parse_pdf, parse_html, parse_docx,
    chunk_recursive,
    scrub_pii_regex, PII_PATTERNS,
    METADATA_FIELDS,
    check_samples_exist,
)

check_samples_exist()
print("Setup ok.")
print(f"  Presidio: installed")
print(f"  OpenAI key set")
print(f"  Qdrant URL: {os.environ['QDRANT_URL'][:40]}...")
print(f"  Sample docs present")
print(f"  Day 1 helpers imported: parse_*, chunk_recursive, scrub_pii_regex, METADATA_FIELDS")

---

## Cell 2 — The 9-field metadata schema

Every chunk gets 9 pieces of metadata attached. The schema is programme-standard
— once you commit to it, downstream code (retrieval, filtering, observability)
can rely on it.

Let's build a chunk with all 9 fields on the sample PDF's page 1.

In [ ]:
from datetime import datetime, timezone

# Print the schema — the fields
print("Metadata schema — 9 fields per chunk:")
for i, field in enumerate(METADATA_FIELDS, 1):
    print(f"  {i}. {field}")

# Build one real chunk with all 9 fields populated
pdf_text = parse_pdf(SAMPLE_PDF)
chunks = chunk_recursive(pdf_text, max_size=400)
first_chunk_text = chunks[0]

chunk_with_metadata = {
    "chunk_id":     "company_policies#0",
    "source":       SAMPLE_PDF.name,
    "doc_type":     "pdf",
    "section_path": "Employee Handbook > Leave Policy",   # from headings
    "page":         1,
    "date":         "2025-01-15",                          # from document metadata
    "language":     "en",
    "version":      "v2.3",
    "ingested_at":  datetime.now(timezone.utc).isoformat(),
    "text":         first_chunk_text,
}

print("\n══ A real chunk with all 9 metadata fields ══\n")
for k, v in chunk_with_metadata.items():
    if k == "text":
        v = v[:80] + "..."
    print(f"  {k:15s}  {v!r}")

**Why 9 fields?** Each earns its place at retrieval time:

| Field | Enables |
|---|---|
| `chunk_id` | Unique reference; deduplication; citations |
| `source` | "Which document did this come from?" |
| `doc_type` | Route retrieval by format (e.g. skip PDFs for a code query) |
| `section_path` | Explainable citations: "from HR > Leave Policy" |
| `page` | Deep-link into PDFs for user verification |
| `date` | Filter by recency: "only current-year policies" |
| `language` | Multi-lingual corpora need this |
| `version` | Retrieve latest revision; audit trail |
| `ingested_at` | Debug drift: "when did this chunk enter the system?" |

**Today's overhead, tomorrow's payoff.** W9 uses filters heavily; without
metadata now, W9's precision lift isn't possible.

---

## Cell 3 — Qdrant filter DSL

Metadata is only useful if you can filter on it at query time. Let's build a
small Qdrant collection with 6 chunks (mixed doc types + dates), then run
**filtered queries**.

In [ ]:
from openai import OpenAI
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance, VectorParams, PointStruct,
    Filter, FieldCondition, MatchValue, Range,
)

openai_client = OpenAI()
qdrant = QdrantClient(url=os.environ["QDRANT_URL"], api_key=os.environ["QDRANT_API_KEY"])

COLLECTION = "wk08_day2_metadata_demo"

# Recreate
try:
    qdrant.delete_collection(COLLECTION)
except Exception:
    pass
qdrant.create_collection(
    collection_name=COLLECTION,
    vectors_config=VectorParams(size=1536, distance=Distance.COSINE),
)

# Six chunks with varied metadata.
# NOTE: for date-range filtering, Qdrant's Range works on NUMERIC fields.
# We store both the ISO date string (for display) and a numeric year (for filtering).
# This is a common real-world pattern: keep both a human-readable form and
# a filter-friendly form of the same value.
sample_chunks = [
    {"id": 0, "text": "Annual leave is 20 days per year.",         "source": "policies.pdf",   "doc_type": "pdf",  "date": "2025-01-15", "year": 2025},
    {"id": 1, "text": "Sick leave is unlimited with doctor's note.", "source": "policies.pdf",   "doc_type": "pdf",  "date": "2025-01-15", "year": 2025},
    {"id": 2, "text": "Remote work requires VP approval.",           "source": "policies.pdf",   "doc_type": "pdf",  "date": "2025-01-15", "year": 2025},
    {"id": 3, "text": "Legacy leave policy: 15 days per year.",      "source": "old_policies.pdf", "doc_type": "pdf", "date": "2022-03-01", "year": 2022},
    {"id": 4, "text": "Product tour: analytics dashboards.",         "source": "landing.html",   "doc_type": "html", "date": "2025-06-20", "year": 2025},
    {"id": 5, "text": "Onboarding: meet your team in week 1.",       "source": "onboarding.docx", "doc_type": "docx", "date": "2024-11-10", "year": 2024},
]

# Embed and upsert
texts = [c["text"] for c in sample_chunks]
vectors = [item.embedding for item in openai_client.embeddings.create(
    model="text-embedding-3-small", input=texts).data]

points = [
    PointStruct(id=c["id"], vector=v, payload={k: c[k] for k in ["text", "source", "doc_type", "date", "year"]})
    for c, v in zip(sample_chunks, vectors)
]
qdrant.upsert(collection_name=COLLECTION, points=points)
print(f"Upserted {len(points)} chunks with metadata.\n")

In [ ]:
def show_hits(results, label):
    print(f"── {label} ──")
    if not results:
        print("  (no matches)\n")
        return
    for h in results:
        p = h.payload
        print(f"  {h.score:.3f}  {p['source']:20s} ({p['doc_type']}, {p['date']})  {p['text'][:50]}")
    print()

# Query with NO filter — pure semantic
query = "How much leave do I get per year?"
q_vec = openai_client.embeddings.create(model="text-embedding-3-small", input=[query]).data[0].embedding

results = qdrant.query_points(collection_name=COLLECTION, query=q_vec, limit=3).points
show_hits(results, f"No filter — Q: {query!r}")

# Now filter: only PDFs
results = qdrant.query_points(
    collection_name=COLLECTION, query=q_vec, limit=3,
    query_filter=Filter(must=[FieldCondition(key="doc_type", match=MatchValue(value="pdf"))]),
).points
show_hits(results, "Filter: doc_type = 'pdf'")

# Now filter: only docs from 2024 or later (uses the numeric 'year' field)
results = qdrant.query_points(
    collection_name=COLLECTION, query=q_vec, limit=3,
    query_filter=Filter(must=[FieldCondition(key="year", range=Range(gte=2024))]),
).points
show_hits(results, "Filter: year >= 2024 (exclude legacy 2022 doc)")

# Cleanup
qdrant.delete_collection(COLLECTION)
print("Cleaned up demo collection.")

**Discussion:**
- Without a filter, the legacy 2022 policy came back — because it matches
  the query semantically. Users would be told the wrong policy.
- The date filter (`date >= 2024-01-01`) excludes legacy. This is a common
  W9 pattern.
- Qdrant's filter DSL supports `must`, `must_not`, `should` (like Elasticsearch),
  and conditions on any payload field. See Qdrant docs for the full grammar.

**Why this matters:** metadata + filters is how you go from 'retrieval sometimes
returns bad results' to 'retrieval respects business rules.'

---

## Cell 4 — Why PII in vector stores is a problem

**Vector stores are search indexes.** Anything you put in them is discoverable
by anyone with query access. If your ingested chunks contain PII, that PII is
one query away from being surfaced.

**Let's watch this happen.** We'll embed a chunk with PII and retrieve it.

In [ ]:
# A chunk with real-looking PII (all fake, but the model doesn't know that)
leaky_chunk = (
    "For questions about payroll, contact Michael Torres in Finance. "
    "His email is michael.torres@acme.com and his employee ID is EMP-04521."
)

# Embed it
vec = openai_client.embeddings.create(
    model="text-embedding-3-small", input=[leaky_chunk]).data[0].embedding

# Build a tiny collection with just this one chunk
LEAK_COLL = "wk08_day2_leak_demo"
try: qdrant.delete_collection(LEAK_COLL)
except Exception: pass

qdrant.create_collection(collection_name=LEAK_COLL,
                        vectors_config=VectorParams(size=1536, distance=Distance.COSINE))
qdrant.upsert(collection_name=LEAK_COLL,
             points=[PointStruct(id=0, vector=vec, payload={"text": leaky_chunk})])

# Query with something innocent-looking
query = "who handles payroll questions?"
q_vec = openai_client.embeddings.create(
    model="text-embedding-3-small", input=[query]).data[0].embedding

hits = qdrant.query_points(collection_name=LEAK_COLL, query=q_vec, limit=1).points

print(f"Q: {query!r}\n")
print("Retrieved chunk:")
print(f"  {hits[0].payload['text']}")
print()
print("The PII (name, email, employee ID) came back in the search result.")
print("Anyone with query access to this collection can retrieve it.")
print("That includes future features you haven't built yet.")

# Cleanup
qdrant.delete_collection(LEAK_COLL)

**The rule:** redact BEFORE you ingest, not after.

Once PII is in the vector store, you can't recall it. You'd have to identify
the offending chunks, delete them, and re-ingest cleaned versions. That's
expensive and error-prone.

**Scrub at ingestion time.** Every time.

---

## Cell 5 — PII detection #1: Regex baseline

**Simplest approach:** regular expressions for common patterns — emails, phone
numbers, employee IDs. Fast, no dependencies, but catches only what you write
patterns for.

The `wk08_pipeline.py` helper has 3 patterns built in.

In [ ]:
# The patterns we have
print("Regex patterns:")
for label, pattern in PII_PATTERNS.items():
    print(f"  {label:8s}  {pattern.pattern}")

print()

# Test on a paragraph with mixed PII
test_text = (
    "Contact Sarah Chen (HR Business Partner) at sarah.chen@acme.com or "
    "+1-555-0142. Her employee ID is EMP-01847. You can also reach Michael "
    "Torres at michael.torres@acme.com."
)

scrubbed, flags = scrub_pii_regex(test_text)

print(f"Original text:\n  {test_text}\n")
print(f"Scrubbed text:\n  {scrubbed}\n")
print(f"PII items detected ({len(flags)}):")
for f in flags:
    print(f"  ✓ {f}")

print("\nMissed: names ('Sarah Chen', 'Michael Torres'). Regex can't do NER.")
print("That's what Presidio brings — next cell.")

---

## Cell 6 — PII detection #2: Presidio NER

**Presidio** uses named entity recognition (NER, backed by spaCy) to find
entities that regex can't — especially **person names, locations, and
organisations**.

It also handles emails, phones, and IDs — with confidence scores per detection.

In [ ]:
# Presidio was imported and asserted in Cell 1
analyzer = AnalyzerEngine()

# Analyze the same text as Cell 5
results = analyzer.analyze(text=test_text, language="en")

print(f"Text:\n  {test_text}\n")
print(f"Presidio detected {len(results)} entities:\n")
print(f"  {'ENTITY_TYPE':<20s} {'SCORE':<7s} {'TEXT':<30s}")
print(f"  {'-'*20} {'-'*7} {'-'*30}")

for r in sorted(results, key=lambda x: x.start):
    entity_text = test_text[r.start:r.end]
    print(f"  {r.entity_type:<20s} {r.score:<7.2f} {entity_text:<30s}")

print()
print("Notice PERSON entities — regex could NEVER find these.")
print("This is why Presidio (or a similar NER tool) is non-negotiable for high-recall PII.")

**Discussion:**
- Did Presidio catch 'Sarah Chen' and 'Michael Torres'? (It should.)
- What about the emails and phone numbers — did they show up too? (Yes, Presidio
  has its own regexes internally, plus NER on top.)
- The `score` column matters — 0.85+ is high confidence, 0.5-0.85 medium, below
  0.5 is worth double-checking manually.

**Presidio isn't perfect.** It misses:
- Non-Latin names (e.g. Chinese, Arabic scripts)
- Novel ID formats (custom internal codes)
- Addresses in unusual layouts

Which is why the three-legs pattern (Cell 8) matters.

---

## Cell 7 — Presidio anonymization

**Detection is half the job.** The other half is doing something with the
detections — usually replacing them with placeholders.

Presidio's `AnonymizerEngine` takes the analyzer output and applies
replacements.

In [ ]:
anonymizer = AnonymizerEngine()

# Analyze then anonymize
results = analyzer.analyze(text=test_text, language="en")
anonymized = anonymizer.anonymize(text=test_text, analyzer_results=results)

print("Before:")
print(f"  {test_text}\n")
print("After anonymization:")
print(f"  {anonymized.text}\n")
print(f"Replacements made: {len(anonymized.items)}")

**Default behaviour:** replace with `<ENTITY_TYPE>` tokens like `<PERSON>`
or `<EMAIL_ADDRESS>`. You can customise operators — hash, encrypt, redact-
with-length-preserved, etc. Presidio docs cover the details.

**For RAG:** the default replacement is fine. The chunk 'contact <PERSON> at
<EMAIL_ADDRESS>' is still retrievable for 'who do I contact?' queries, but
the actual identity isn't leaked.

---

## Cell 8 — The three legs: regex + Presidio + manual audit

**No single tool catches everything.** Production PII pipelines have three legs:

1. **Regex** — fast; catches obvious patterns (emails, phones, IDs); low recall on novel formats
2. **Presidio (or similar NER)** — catches names, orgs, locations; higher recall; slower; misses non-Latin scripts
3. **Manual audit** — human eyes on a sample of ingested chunks; catches what both missed

Let's build a hybrid `scrub_pii` that runs both regex and Presidio, then
returns everything flagged.

In [ ]:
def scrub_pii_hybrid(text: str) -> tuple[str, list[dict]]:
    """Regex + Presidio combined. Returns (scrubbed_text, flags).
    
    flags is a list of dicts: {source, entity_type, matched_text, score}
    """
    flags = []
    
    # Leg 1: regex scrub (from Day 1 helper)
    scrubbed, regex_flags = scrub_pii_regex(text)
    for f in regex_flags:
        entity_type, matched = f.split(": ", 1)
        flags.append({"source": "regex", "entity_type": entity_type,
                     "matched_text": matched, "score": 1.0})
    
    # Leg 2: Presidio on the ALREADY-regex-scrubbed text
    # (avoids double-flagging emails/phones that regex already redacted)
    results = analyzer.analyze(text=scrubbed, language="en")
    if results:
        anonymized = anonymizer.anonymize(text=scrubbed, analyzer_results=results)
        scrubbed = anonymized.text
        for r in results:
            flags.append({"source": "presidio", "entity_type": r.entity_type,
                         "matched_text": text[r.start:r.end], "score": r.score})
    
    return scrubbed, flags

# Run on the test text
scrubbed, flags = scrub_pii_hybrid(test_text)

print(f"Original:\n  {test_text}\n")
print(f"Scrubbed:\n  {scrubbed}\n")
print(f"Total flags: {len(flags)}\n")

print(f"  {'SOURCE':<10s} {'ENTITY':<20s} {'SCORE':<7s} {'MATCHED':<25s}")
print(f"  {'-'*10} {'-'*20} {'-'*7} {'-'*25}")
for f in flags:
    print(f"  {f['source']:<10s} {f['entity_type']:<20s} {f['score']:<7.2f} {f['matched_text']:<25s}")

print("\n**Leg 3 (audit):** in production, human eyes review 10-20 random chunks")
print("per ingestion run. Look for what neither tool caught. Log gaps in an audit doc.")

**Real-world reality check:**
- Regex + Presidio is a strong pairing — usually 90-95% recall on English text
- The remaining 5-10% needs human review for high-stakes data
- **Don't skip audit.** It's tedious but catches novel PII patterns your tools
  don't know about (custom ID formats, unusual addresses, initials-only names)

---

## Cell 9 — Data residency: name your stack

**Every RAG system spans jurisdictions.** Your embeddings live somewhere, your
LLM call goes somewhere, your vector store is hosted somewhere. Each hop crosses
borders. GDPR, DPDP, and similar regulations care about this.

**W8's residency requirement:** you can articulate your stack in one sentence.

Not a deep-dive — that's W28. Just name it.

In [ ]:
def describe_stack() -> None:
    """Print a one-sentence description of where your data lives."""
    qdrant_url = os.environ.get("QDRANT_URL", "unknown")
    
    # Detect Qdrant Cloud region from URL if possible
    if "aws" in qdrant_url.lower():
        qdrant_region = "AWS (region in URL)"
    elif "gcp" in qdrant_url.lower():
        qdrant_region = "Google Cloud (region in URL)"
    elif "azure" in qdrant_url.lower():
        qdrant_region = "Azure (region in URL)"
    elif "localhost" in qdrant_url or "127.0.0.1" in qdrant_url:
        qdrant_region = "local machine"
    else:
        qdrant_region = "Qdrant Cloud (check dashboard for region)"
    
    print("═══ YOUR DATA RESIDENCY STACK ═══\n")
    print(f"  Embeddings computed by: OpenAI API (US)")
    print(f"  Vectors stored at:      {qdrant_region}")
    print(f"                          URL: {qdrant_url[:50]}...")
    print(f"  LLM inference at:       OpenAI API (US)")
    print(f"  App layer:              wherever you run this notebook")
    print()
    print("One-sentence articulation:")
    print("  \"My embeddings are computed and my LLM calls are served by OpenAI (US);")
    print("   my vectors sit in Qdrant Cloud; my source documents live locally.\"\n")
    print("Jurisdictions crossed: US (OpenAI), + Qdrant region, + wherever your app runs.")
    print("If your users' data is EU-residents' data → GDPR applies. Log it in your ADR.")

describe_stack()

**That's the W8 residency bar.** Not a full compliance analysis — just being
able to answer 'where does my data go?' for your capstone stack.

W28 deep-dives residency, retention, right-to-be-forgotten, and audit logging.
For now, put your stack sentence in your ADR (Track B step 4).

---

## Cell 10 — Wire it together: mini-pipeline on ONE document

**The synthesis moment.** Take one sample document, run it end-to-end through:
1. Parse (Day 1)
2. Structure-aware chunk (Day 1)
3. Scrub PII (Cell 8 — hybrid regex + Presidio)
4. Attach metadata (Cell 2 — 9 fields)
5. Print the final list of ingestion-ready chunks

We do NOT push these to Qdrant here — that's Track B on the capstone corpus.
Today's goal is to see the shape of an ingested chunk.

In [ ]:
from datetime import datetime, timezone

def ingest_one_pdf(path: Path) -> list[dict]:
    """End-to-end ingestion of a single PDF: parse → chunk → scrub → enrich.
    
    Returns a list of dicts ready for Qdrant upsert.
    """
    # Step 1: parse
    text = parse_pdf(path)
    
    # Step 2: chunk (recursive is the sane default from Day 1)
    chunk_texts = chunk_recursive(text, max_size=400)
    
    # Step 3: scrub PII from each chunk
    ingested_at = datetime.now(timezone.utc).isoformat()
    doc_stem = path.stem  # e.g. 'company_policies'
    
    ingested = []
    for idx, raw_text in enumerate(chunk_texts):
        scrubbed, flags = scrub_pii_hybrid(raw_text)
        
        # Step 4: enrich with metadata
        chunk = {
            "chunk_id":         f"{doc_stem}#{idx}",
            "source":           path.name,
            "doc_type":         path.suffix.lstrip(".").lower(),
            "section_path":     doc_stem,  # placeholder; structure-aware would fill this properly
            "page":             None,       # PDF-parse could extract this per-chunk if needed
            "date":             "2025-01-15",  # in reality, extract from PDF metadata
            "language":         "en",
            "version":          "v1",
            "ingested_at":      ingested_at,
            "text":             scrubbed,
            "pii_flags_count":  len(flags),
        }
        ingested.append(chunk)
    
    return ingested

# Run on the sample PDF
chunks = ingest_one_pdf(SAMPLE_PDF)

print(f"Ingested {SAMPLE_PDF.name} → {len(chunks)} chunks\n")
print(f"Total PII items scrubbed: {sum(c['pii_flags_count'] for c in chunks)}\n")
print("══ Sample of ingested chunks ══\n")

for c in chunks[:3]:  # first 3 for brevity
    print(f"[{c['chunk_id']}]")
    print(f"  source: {c['source']}, doc_type: {c['doc_type']}, pii_flags: {c['pii_flags_count']}")
    print(f"  text:   {c['text'][:120]}...")
    print()

# Show a chunk that has PII scrubbed (should be the HR contact one)
pii_chunks = [c for c in chunks if c['pii_flags_count'] > 0]
if pii_chunks:
    print(f"══ Example of a scrubbed chunk (had {pii_chunks[0]['pii_flags_count']} PII items) ══")
    print(f"  {pii_chunks[0]['text']}")
    print("\n  Notice: emails, phones, names all replaced with placeholders.")
    print("  The chunk is still semantically useful for retrieval, but the PII is gone.")

**That's what a chunk looks like at ingestion time.** All 9 metadata fields
populated, PII scrubbed, text ready for embedding + upsert to Qdrant.

**Not shown here (Track B does):**
- Wrapping `ingest_one_pdf` into `ingest_corpus(dir)` that dispatches by extension
- Embedding the scrubbed text
- Upserting to Qdrant with the full metadata payload
- Running the golden set against the new collection

**Track B is where this shape becomes real capstone code.**

---

## Cell 11 — Wrap: what you built across two days

**Day 1 — Parsing + chunking:**
1. PyMuPDF (fast), pdfplumber (tables), BeautifulSoup (strip nav/footer), python-docx (styles)
2. The scanned-PDF trap — empty text, silent failure
3. Four chunking strategies compared; structure-aware wins when documents have headings

**Day 2 — Metadata + PII + wiring:**
4. 9-field metadata schema — each field earns retrieval-time payoff
5. Qdrant filter DSL — filter by doc_type, date, source, etc.
6. PII in vector stores is discoverable — scrub at ingestion, not after
7. Regex + Presidio + audit — three legs for high-recall PII
8. Data residency — you can articulate your stack in one sentence
9. Mini-pipeline: parse → chunk → scrub → enrich → ingestion-ready chunk

**Track B (take-home): apply this to your capstone corpus.**

Open `AI-RAG_W8_Application_Growth_Guide.md`. You will:
- Add `src/ingest/pipeline.py` (~120 lines) — the full corpus ingester
- Create Qdrant collection `capstone_chunks_v2` with metadata payload
- Re-ingest your capstone corpus
- Verify PII scrubbing on 10 random chunks (audit doc)
- Re-run golden set, compare wk8 vs wk7 snapshot
- Update ADR with ingestion decisions

About 3 hours self-paced.

---

**Heads-up for W9:** the difficulty heatmap flags W9 as the **first DANGER
ZONE** — hybrid search + reranking + query rewriting all at once. Plan extra
lab time. But your W8 metadata will make W9's filtered retrieval powerful.